# 2. Feature Engineering
Takes the cleaned output of `dataProcessing.ipynb` and builds the model-ready dataset.

**Framing:** for every GPS point we pick its nearest road arcs (the *candidates*), compute features describing
how well each candidate fits the point, and label whether the candidate is on the true route.
Every one of the 5 models learns P(candidate is on route); the HMM/Viterbi step later uses that as its emission score.

**Steps:** imports -> config -> load -> kinematic features -> network features -> candidate search -> candidate features -> labels -> run all subjects -> sanity checks -> scale & save

## 2.1 Imports

In [1]:
import json
import os
import pickle

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 2.2 Configuration
- `K_CAND`: candidate arcs kept per GPS point
- `SEARCH_R`: candidate search radius in metres (99% of points lie within ~130 m of their true route)
- `DENSIFY_M`: vertex spacing of the spatial index
- `STATIONARY_M`: if the vehicle moves less than this, its heading is unreliable

In [2]:
OUT_DIR = "processed"
K_CAND = 10
SEARCH_R = 120.0
DENSIFY_M = 15.0
STATIONARY_M = 3.0
MAX_SPEED = 60.0

FEATURES = [
    "dist", "dist_rank", "dist_minus_min", "n_within_50m", "frac_along", "arc_len",
    "head_diff_cos", "head_valid", "arc_sin", "arc_cos", "end_outdeg", "start_indeg",
    "dist_to_start", "dist_to_end", "speed", "dt", "accel", "turn_rate", "seg_pos",
]

## 2.3 Load cleaned data
Reads the intermediate file written by `dataProcessing.ipynb`.

In [3]:
with open(os.path.join(OUT_DIR, "interim", "interim.pkl"), "rb") as f:
    interim = pickle.load(f)

points, networks, split_ids = interim["points"], interim["networks"], interim["split_ids"]
print({k: len(v) for k, v in split_ids.items()}, "| points:", len(points))
points.head(3)

{'train': 62, 'val': 16, 'test': 10, 'dropped': 2} | points: 201599


,point_idx,lon,lat,t,x,y,seg_id,dt,subject_id
0,0,8.595203,49.844053,1.0,-1274.970314,-22549.553996,0,0.0,00000001
1,1,8.595294,49.844441,2.0,-1268.472446,-22506.410305,0,1.0,00000001
2,2,8.595352,49.844883,3.0,-1264.330947,-22457.262080,0,1.0,00000001


## 2.4 Kinematic features (per GPS point)
Computed inside each track segment only (never across a time gap):
`speed`, `accel`, `heading` (central difference of 3 points), `head_valid` (0 when stationary),
`turn_rate`, and `seg_pos` (0..1 position inside the segment, tells sequence models where a trace starts/ends).

In [4]:
def add_kinematics(df):
    df = df.copy()
    n = len(df)
    xy = df[["x", "y"]].to_numpy()
    seg = df["seg_id"].to_numpy()
    dt = df["dt"].to_numpy()
    new_seg = np.r_[True, seg[1:] != seg[:-1]]

    step = np.r_[0.0, np.linalg.norm(np.diff(xy, axis=0), axis=1)]
    step[new_seg] = 0.0
    with np.errstate(divide="ignore", invalid="ignore"):
        speed = np.where(dt > 0, step / dt, 0.0)
    speed = np.clip(speed, 0, MAX_SPEED)
    with np.errstate(divide="ignore", invalid="ignore"):
        accel = np.r_[0.0, np.diff(speed)] / np.where(dt > 0, dt, np.nan)
    accel[new_seg | np.r_[True, new_seg[:-1]]] = 0.0
    accel = np.clip(np.nan_to_num(accel), -10, 10)

    idx = np.arange(n)
    prev_i = np.where(new_seg, idx, idx - 1)
    next_i = np.minimum(idx + 1, n - 1)
    next_i = np.where(seg[next_i] != seg, idx, next_i)
    d = xy[next_i] - xy[prev_i]
    heading = np.arctan2(d[:, 1], d[:, 0])
    head_valid = (np.linalg.norm(d, axis=1) > STATIONARY_M).astype(float)

    with np.errstate(divide="ignore", invalid="ignore"):
        turn = np.r_[0.0, np.abs(np.angle(np.exp(1j * np.diff(heading))))] / np.where(dt > 0, dt, np.nan)
    turn_rate = np.clip(np.nan_to_num(turn), 0, np.pi)
    turn_rate[new_seg] = 0.0

    g = df.groupby("seg_id")["t"]
    span = (g.transform("max") - g.transform("min")).replace(0, 1)

    df["dist_prev"], df["speed"], df["accel"] = step, speed, accel
    df["heading"], df["head_valid"], df["turn_rate"] = heading, head_valid, turn_rate
    df["seg_pos"] = (df["t"] - g.transform("min")) / span
    return df

## 2.5 Network features (per road arc)
Arc bearing (as sin/cos so 359 deg and 1 deg are close) and junction complexity:
`end_outdeg` = exits at the arc's end node, `start_indeg` = entries at its start node. Only valid arcs are counted.

In [5]:
def add_network_features(net):
    u, v = net["arcs"][:, 0], net["arcs"][:, 1]
    A, B = net["node_xy"][u], net["node_xy"][v]
    n_nodes = len(net["node_xy"])
    net["ang"] = np.arctan2(B[:, 1] - A[:, 1], B[:, 0] - A[:, 0])
    net["outdeg"] = np.bincount(u[net["valid"]], minlength=n_nodes)
    net["indeg"] = np.bincount(v[net["valid"]], minlength=n_nodes)
    net["A"], net["B"] = A, B
    return net

## 2.6 Candidate search
1. Index: densify every valid arc into vertices every `DENSIFY_M` m and put them in a KD-tree.
2. For each GPS point take all arcs within `SEARCH_R`; if fewer than `K_CAND` are found (sparse area) fall back to the 64 nearest vertices.
3. Compute the exact point-to-segment distance and keep the `K_CAND` closest arcs.

In [6]:
def build_index(net):
    ids = np.flatnonzero(net["valid"])
    n = np.maximum(np.ceil(net["arc_len"][ids] / DENSIFY_M).astype(int), 1) + 1
    arc_of = np.repeat(ids, n)
    t = (np.arange(n.sum()) - np.repeat(np.cumsum(n) - n, n)) / np.repeat(n - 1, n)
    verts = net["A"][arc_of] + (net["B"][arc_of] - net["A"][arc_of]) * t[:, None]
    return cKDTree(verts), arc_of


def point_to_arcs(p, arc_ids, net):
    A, B = net["A"][arc_ids], net["B"][arc_ids]
    AB = B - A
    t = np.clip(((p - A) * AB).sum(1) / (AB ** 2).sum(1), 0, 1)
    return np.linalg.norm(p - (A + AB * t[:, None]), axis=1), t


def search_candidates(pts, net):
    tree, arc_of = build_index(net)
    P = pts[["x", "y"]].to_numpy()
    balls = tree.query_ball_point(P, r=SEARCH_R + DENSIFY_M / 2)
    out_pt, out_arc, out_d, out_t, out_rank = [], [], [], [], []
    for i, (p, hits) in enumerate(zip(P, balls)):
        cand = np.unique(arc_of[hits]) if len(hits) else np.array([], int)
        if len(cand) < K_CAND:
            _, nn = tree.query(p, k=min(64, tree.n))
            cand = np.unique(np.r_[cand, arc_of[np.atleast_1d(nn)]]).astype(int)
        d, t = point_to_arcs(p, cand, net)
        o = np.argsort(d)[:K_CAND]
        out_pt.append(np.full(len(o), i)); out_arc.append(cand[o]); out_d.append(d[o]); out_t.append(t[o])
        out_rank.append(np.arange(len(o)))
    return pd.DataFrame({"pt": np.concatenate(out_pt), "arc_id": np.concatenate(out_arc),
                         "dist": np.concatenate(out_d), "frac_along": np.concatenate(out_t),
                         "dist_rank": np.concatenate(out_rank)})

## 2.7 Candidate features (per point-arc pair)
How well does this arc explain this GPS point?
distance to arc, rank among candidates, gap to the closest candidate, how crowded the area is, position along the arc,
arc length, distance to arc start/end, heading agreement (`+1` same direction, `-1` opposite, `0` unknown),
arc bearing, junction degree, plus the point's own kinematics copied onto each row.

In [7]:
def add_candidate_features(c, pts, net):
    arc = c["arc_id"].to_numpy()
    g = c.groupby("pt")["dist"]
    c["dist_minus_min"] = c["dist"] - g.transform("min")
    c["n_within_50m"] = g.transform(lambda s: (s <= 50).sum())
    c["arc_len"] = net["arc_len"][arc]
    c["dist_to_start"] = c["frac_along"] * c["arc_len"]
    c["dist_to_end"] = (1 - c["frac_along"]) * c["arc_len"]
    ang = net["ang"][arc]
    c["arc_sin"], c["arc_cos"] = np.sin(ang), np.cos(ang)
    c["end_outdeg"] = net["outdeg"][net["arcs"][arc, 1]]
    c["start_indeg"] = net["indeg"][net["arcs"][arc, 0]]

    pt = c["pt"].to_numpy()
    hv = pts["head_valid"].to_numpy()[pt]
    diff = np.angle(np.exp(1j * (pts["heading"].to_numpy()[pt] - ang)))
    c["head_valid"] = hv
    c["head_diff_cos"] = np.cos(diff) * hv
    c["head_diff_deg"] = np.degrees(np.abs(diff)) * hv     # only used for labelling, not a model input
    for col in ["speed", "dt", "accel", "turn_rate", "seg_pos"]:
        c[col] = pts[col].to_numpy()[pt]
    return c

## 2.8 Labels
- `on_route` (**main target**): 1 if the candidate arc is part of the true route (duplicate arcs resolved through `canon_id`).
- `is_true_arc`: the single best on-route candidate for each point (nearest, heading breaks ties). Useful for per-point accuracy and for evaluating decoding.
- `point_idx`, `seg_id`, `subject_id` are kept so rows can be traced back to the raw files and grouped for CV.

In [8]:
def add_labels(c, pts, net):
    true_set = np.unique(net["canon_id"][net["route"]])
    c["on_route"] = np.isin(net["canon_id"][c["arc_id"].to_numpy()], true_set).astype(int)
    score = np.where(c["on_route"] == 1, c["dist"] + 0.1 * c["head_diff_deg"], np.inf)
    best = pd.Series(score).groupby(c["pt"].to_numpy()).transform("min").to_numpy()
    c["is_true_arc"] = (np.isfinite(score) & (score == best)).astype(int)
    pt = c["pt"].to_numpy()
    c["point_idx"] = pts["point_idx"].to_numpy()[pt]
    c["seg_id"] = pts["seg_id"].to_numpy()[pt]
    c["subject_id"] = pts["subject_id"].iloc[0]
    return c.drop(columns="pt")

## 2.9 Run for all subjects
Applies 2.4 to 2.8 subject by subject and stores the result under its split (train / val / test).

In [9]:
sid_to_split = {sid: sp for sp in ["train", "val", "test"] for sid in split_ids[sp]}
pts_out = {"train": [], "val": [], "test": []}
cand_out = {"train": [], "val": [], "test": []}
recall_log = []

for n, (sid, grp) in enumerate(points.groupby("subject_id")):
    sp = sid_to_split[sid]
    net = add_network_features(networks[sid])
    pts = add_kinematics(grp.reset_index(drop=True))
    c = search_candidates(pts, net)
    c = add_candidate_features(c, pts, net)
    c = add_labels(c, pts, net)
    pts_out[sp].append(pts)
    cand_out[sp].append(c)
    recall_log.append(dict(subject_id=sid, split=sp, n_points=len(pts), n_rows=len(c),
                           cand_recall=c.groupby("point_idx")["on_route"].max().mean()))
    if (n + 1) % 10 == 0:
        print(f"{n + 1} subjects done")

pts_out = {k: pd.concat(v, ignore_index=True) for k, v in pts_out.items()}
cand_out = {k: pd.concat(v, ignore_index=True) for k, v in cand_out.items()}
recall_df = pd.DataFrame(recall_log)

10 subjects done
20 subjects done
30 subjects done
40 subjects done
50 subjects done
60 subjects done
70 subjects done
80 subjects done


## 2.10 Sanity checks
- **candidate recall**: share of GPS points that have at least one on-route arc among their candidates (upper bound for any model)
- no missing values in the feature columns
- class balance of `on_route`

In [10]:
rows = []
for sp in ["train", "val", "test"]:
    c = cand_out[sp]
    rows.append(dict(split=sp, subjects=c.subject_id.nunique(), points=len(pts_out[sp]), candidate_rows=len(c),
                     nan_in_features=int(c[FEATURES].isna().sum().sum()), pos_rate=round(c.on_route.mean(), 3),
                     mean_recall=round(recall_df[recall_df.split == sp].cand_recall.mean(), 4),
                     min_recall=round(recall_df[recall_df.split == sp].cand_recall.min(), 4)))
pd.DataFrame(rows)

,split,subjects,points,candidate_rows,nan_in_features,pos_rate,mean_recall,min_recall
0,train,62,150182,1347223,0,0.379,0.9935,0.8710
1,val,16,32259,313307,0,0.332,0.9953,0.9595
2,test,10,19158,171447,0,0.401,0.9991,0.9960


In [11]:
cand_out["train"][FEATURES].describe().T[["mean", "std", "min", "max"]].round(3)

,mean,std,min,max
dist,44.060,46.589,0.00,455.395
dist_rank,4.241,2.846,0.00,9.000
dist_minus_min,37.450,45.670,0.00,454.114
n_within_50m,6.634,3.205,0.00,10.000
frac_along,0.499,0.443,0.00,1.000
arc_len,116.516,225.733,0.07,4667.684
head_diff_cos,0.028,0.806,-1.00,1.000
head_valid,0.967,0.180,0.00,1.000
arc_sin,0.011,0.676,-1.00,1.000
arc_cos,0.003,0.737,-1.00,1.000


## 2.11 Scale and save
`StandardScaler` is **fit on the train split only** and applied to val and test (no leakage).
Per split we save `model_data.npz` (scaled `X`, labels, subject groups, ids), the readable `candidates.csv.gz` and `points.csv.gz`.
Tree models do not need scaling, but it is harmless; SVM, logistic regression and neural nets do.

In [12]:
scaler = StandardScaler().fit(cand_out["train"][FEATURES].to_numpy(np.float64))
with open(os.path.join(OUT_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

for sp in ["train", "val", "test"]:
    c = cand_out[sp]
    d = os.path.join(OUT_DIR, sp)
    os.makedirs(d, exist_ok=True)
    np.savez_compressed(
        os.path.join(d, "model_data.npz"),
        X=scaler.transform(c[FEATURES].to_numpy(np.float64)).astype(np.float32),
        y=c["on_route"].to_numpy(np.int8), y_true_arc=c["is_true_arc"].to_numpy(np.int8),
        groups=c["subject_id"].astype(str).to_numpy(), point_idx=c["point_idx"].to_numpy(),
        seg_id=c["seg_id"].to_numpy(), arc_id=c["arc_id"].to_numpy(), feature_names=np.array(FEATURES))
    c.to_csv(os.path.join(d, "candidates.csv.gz"), index=False)
    pts_out[sp].to_csv(os.path.join(d, "points.csv.gz"), index=False)

with open(os.path.join(OUT_DIR, "meta.json"), "w") as f:
    json.dump(dict(features=FEATURES, k_cand=K_CAND, search_radius_m=SEARCH_R, splits=split_ids), f, indent=1)
print("saved to", OUT_DIR, "->", os.listdir(OUT_DIR))

saved to processed -> ['interim', 'meta.json', 'scaler.pkl', 'subject_report.csv', 'test', 'train', 'val']
